# Mechanism where classes differ strongly in size

**Computes:** Share of the largest class kept in one cluster and clusters left empty, the representative instances, decisions on the document images, footprint widths on natural images, and prototype errors on the five synthetic scenarios.

**Inputs:** results of 08 and 10, UCI data, BSDS500, DIBCO images.  
**Outputs:** `results/11/results_11.json` (the report is printed in the notebook).


In [ ]:
import os, sys, time, json, pickle, platform, subprocess, glob, urllib.request
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
DATASETS = None
RUN_STAMP = time.strftime("%Y%m%d-%H%M%S")

RESULTS_DIR = os.environ.get("RESULTS_DIR", "results")
OUTDIR = os.path.join(RESULTS_DIR, "11")
OUTDIR = os.path.abspath(OUTDIR)
for sub in ("data", "reports"):
    os.makedirs(os.path.join(OUTDIR, sub), exist_ok=True)
import numpy as np, scipy, sklearn
print(f"colab {IN_COLAB} | python {platform.python_version()} | numpy {np.__version__} | scipy {scipy.__version__} | sklearn {sklearn.__version__}")
print(f"output dir: {OUTDIR}   run stamp: {RUN_STAMP}")
BASE = os.path.dirname(OUTDIR)
DIR_06 = os.environ.get("DIR_06", os.path.join(BASE, "06"))
DIR_07 = os.environ.get("DIR_07", os.path.join(BASE, "07"))
J08_PATH = os.environ.get("J08", os.path.join(BASE, "08", "results_08.json"))
J10_PATH = os.environ.get("J10", os.path.join(BASE, "10", "results_10.json"))
DATA_10 = os.environ.get("DATA_10", os.path.join(BASE, "10", "data"))
for p in (J08_PATH, J10_PATH):
    print(("found   " if os.path.exists(p) else "MISSING ") + p)
assert os.path.exists(J08_PATH), "results_08.json not found: run 08 first"
J08 = json.load(open(J08_PATH)); J10 = json.load(open(J10_PATH)) if os.path.exists(J10_PATH) else None
LOG = {}

In [ ]:
import numpy as np
from itertools import permutations

EPS = 1e-12

def fcm_membership(D, m):
    p = 2.0 / (m - 1.0)
    z = -p * np.log(np.maximum(D, EPS))
    z -= z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def hr_bounds(D, m1, m2):
    C = D.shape[1]
    U1, U2 = fcm_membership(D, m1), fcm_membership(D, m2)
    s = 1.0 / (D[:, :, None] / np.maximum(D[:, None, :], EPS)).sum(axis=2)
    cond = s < 1.0 / C
    upper = np.where(cond, U1, U2)
    lower = np.where(cond, U2, U1)
    return lower, upper

def km_interval_1d(x, wl, wu):
    o = np.argsort(x, kind="stable"); xs = x[o]; l = wl[o]; u = wu[o]
    n = len(x)
    cl, cu = np.cumsum(l), np.cumsum(u)
    clx, cux = np.cumsum(l * xs), np.cumsum(u * xs)
    Tl, Tu, Tlx, Tux = cl[-1], cu[-1], clx[-1], cux[-1]
    k = np.arange(n + 1)
    Lk = np.concatenate([[0.0], cl]); Uk = np.concatenate([[0.0], cu])
    Lxk = np.concatenate([[0.0], clx]); Uxk = np.concatenate([[0.0], cux])
    vR_all = (Lxk + (Tux - Uxk)) / np.maximum(Lk + (Tu - Uk), EPS)
    vL_all = (Uxk + (Tlx - Lxk)) / np.maximum(Uk + (Tl - Lk), EPS)
    kR, kL = int(np.argmax(vR_all)), int(np.argmin(vL_all))
    selR = np.zeros(n, bool); selL = np.zeros(n, bool)
    rank = np.empty(n, int); rank[o] = np.arange(n)
    selR[:] = rank >= kR
    selL[:] = rank < kL
    return float(vL_all[kL]), float(vR_all[kR]), selL, selR

def it2fcm_update(X, lower, upper, mc):
    N, M = X.shape; C = lower.shape[1]
    V = np.zeros((C, M)); uL = np.zeros((N, C)); uR = np.zeros((N, C))
    wl, wu = lower ** mc, upper ** mc
    for j in range(C):
        accL = np.zeros(N); accR = np.zeros(N)
        for l in range(M):
            vL, vR, selL, selR = km_interval_1d(X[:, l], wl[:, j], wu[:, j])
            V[j, l] = 0.5 * (vL + vR)
            accL += np.where(selL, upper[:, j], lower[:, j])
            accR += np.where(selR, upper[:, j], lower[:, j])
        uL[:, j], uR[:, j] = accL / M, accR / M
    return V, uL, uR

def dist(X, V):
    return np.maximum(np.linalg.norm(X[:, None, :] - V[None, :, :], axis=2), EPS)

def run_fcm(X, V0, m, iters=300, tol=1e-7):
    V = V0.copy()
    for _ in range(iters):
        U = fcm_membership(dist(X, V), m); W = U ** m
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    U = fcm_membership(dist(X, V), m)
    return V, U.argmax(1)

def run_it2fcm(X, V0, m1, m2, mc=None, iters=300, tol=1e-7):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds(dist(X, V), m1, m2)
        Vn, uL, uR = it2fcm_update(X, lo, up, mc)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds(dist(X, V), m1, m2)
    _, uL, uR = it2fcm_update(X, lo, up, mc)
    hard_hr = (0.5 * (uL + uR)).argmax(1)
    hard_nearest = dist(X, V).argmin(1)
    return V, hard_hr, hard_nearest

def accuracy(labels, pred, C):
    best = 0.0
    for perm in permutations(range(C)):
        best = max(best, float(np.mean(np.array(perm)[pred] == labels)))
    return best

def centre_error(V, centres):
    C = len(centres); best = np.inf
    for perm in permutations(range(C)):
        best = min(best, float(np.mean(np.linalg.norm(V[list(perm)] - centres, axis=1))))
    return best

def init_pp(X, C, rng):
    idx = [rng.integers(len(X))]
    for _ in range(1, C):
        d2 = np.min(((X[:, None, :] - X[idx][None]) ** 2).sum(2), axis=1)
        idx.append(rng.choice(len(X), p=d2 / d2.sum()))
    return X[idx].copy()

def scenario(name, seed):
    r = np.random.default_rng(seed)
    if name == "equal":
        c = np.array([[0.0, 0.0], [4.0, 0.0]]); n = (100, 100); s = (1.0, 1.0)
        X = np.vstack([r.normal(c[k], s[k], size=(n[k], 2)) for k in range(2)])
    elif name == "volume":
        c = np.array([[0.0, 0.0], [5.0, 0.0]]); n = (100, 100); half = (2.5, 0.6)
        X = np.vstack([c[k] + r.uniform(-half[k], half[k], size=(n[k], 2)) for k in range(2)])
    elif name == "noise":
        c = np.array([[0.0, 0.0], [0.0, 4.0]]); n = (100, 100)
        X = np.vstack([r.normal(c[0], 1.0, size=(n[0], 2)), r.normal(c[1], 1.0, size=(n[1], 2)),
                       r.normal(c[0], 3.0, size=(40, 2))])
        y = np.r_[np.zeros(n[0]), np.ones(n[1]), np.zeros(40)].astype(int)
        return X, y, c, 2
    elif name == "density":
        c = np.array([[0.0, 0.0], [3.5, 0.0]]); n = (200, 30); s = (1.0, 1.0)
        X = np.vstack([r.normal(c[k], s[k], size=(n[k], 2)) for k in range(2)])
    elif name == "volume3":
        c = np.array([[0.0, 0.0], [5.0, 0.0], [2.5, 4.0]]); n = (100, 100, 100); s = (1.4, 0.4, 0.8)
        X = np.vstack([r.normal(c[k], s[k], size=(n[k], 2)) for k in range(3)])
    else:
        raise ValueError(name)
    y = np.concatenate([np.full(n[k], k) for k in range(len(n))])
    return X, y, c, len(n)

from scipy.optimize import linear_sum_assignment
def accuracy_h(labels, pred, C):
    M = np.zeros((C, C))
    np.add.at(M, (labels, pred), 1)
    r, c = linear_sum_assignment(-M)
    return float(M[r, c].sum() / len(labels))

def align(V, ref):
    Dm = np.linalg.norm(V[:, None] - ref[None], axis=2)
    r, c = linear_sum_assignment(Dm)
    out = np.zeros_like(ref); out[c] = V[r]
    return out

def centre_error_h(V, centres):
    return float(np.mean(np.linalg.norm(align(V, centres) - centres, axis=1)))

def real_dataset(name):
    from sklearn import datasets
    d = {"iris": datasets.load_iris, "wine": datasets.load_wine, "breast_cancer": datasets.load_breast_cancer}[name]()
    X = d.data.astype(float); X = (X - X.mean(0)) / X.std(0)
    y = d.target.astype(int)
    return X, y, len(np.unique(y))

def run_fcm_decoupled(X, V0, ma, mb, iters=300, tol=1e-7):
    V = V0.copy()
    for _ in range(iters):
        U = fcm_membership(dist(X, V), ma); W = U ** mb
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    return V, dist(X, V).argmin(1)

def it2fcm_update_fast(X, order, lower, upper, mc):
    N, M = X.shape; C = lower.shape[1]
    Xs = np.take_along_axis(X, order, axis=0)
    rank = np.empty_like(order); np.put_along_axis(rank, order, np.arange(N)[:, None].repeat(M, 1), axis=0)
    V = np.zeros((C, M)); uL = np.zeros((N, C)); uR = np.zeros((N, C))
    z = np.zeros((1, M))
    for j in range(C):
        wl = (lower[:, j] ** mc)[order]; wu = (upper[:, j] ** mc)[order]
        cl, cu = np.cumsum(wl, 0), np.cumsum(wu, 0)
        clx, cux = np.cumsum(wl * Xs, 0), np.cumsum(wu * Xs, 0)
        Lk, Uk = np.vstack([z, cl]), np.vstack([z, cu]); Lxk, Uxk = np.vstack([z, clx]), np.vstack([z, cux])
        Tl, Tu, Tlx, Tux = cl[-1], cu[-1], clx[-1], cux[-1]
        vR_all = (Lxk + (Tux - Uxk)) / np.maximum(Lk + (Tu - Uk), EPS)
        vL_all = (Uxk + (Tlx - Lxk)) / np.maximum(Uk + (Tl - Lk), EPS)
        kR, kL = vR_all.argmax(0), vL_all.argmin(0)
        vR = vR_all[kR, np.arange(M)]; vL = vL_all[kL, np.arange(M)]
        V[j] = 0.5 * (vL + vR)
        selR = rank >= kR[None, :]; selL = rank < kL[None, :]
        up, lo = upper[:, j][:, None], lower[:, j][:, None]
        uR[:, j] = np.where(selR, up, lo).mean(1); uL[:, j] = np.where(selL, up, lo).mean(1)
    return V, uL, uR

def run_it2fcm_fast(X, V0, m1, m2, mc=None, iters=300, tol=1e-7, order=None):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    order = np.argsort(X, axis=0, kind="stable") if order is None else order
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds(dist(X, V), m1, m2)
        Vn, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds(dist(X, V), m1, m2)
    _, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
    return V, (0.5 * (uL + uR)).argmax(1), dist(X, V).argmin(1)

def real_dataset2(name, data_dir="data"):
    import os, pandas as pd
    if name in ("iris", "wine", "wdbc"):
        return real_dataset({"wdbc": "breast_cancer"}.get(name, name))
    if name == "sonar":
        d = pd.read_csv(os.path.join(data_dir, "sonar.csv"), header=None)
        X = d.iloc[:, :-1].to_numpy(float); y = (d.iloc[:, -1] == "M").to_numpy(int)
    elif name == "pima":
        d = pd.read_csv(os.path.join(data_dir, "pima-indians-diabetes.csv"), header=None)
        X = d.iloc[:, :-1].to_numpy(float); y = d.iloc[:, -1].to_numpy(int)
    elif name == "wbc":
        d = pd.read_csv(os.path.join(data_dir, "breast-cancer-wisconsin.data"), header=None, na_values="?").dropna()
        X = d.iloc[:, 1:-1].to_numpy(float); y = (d.iloc[:, -1] == 4).to_numpy(int)
    else:
        raise ValueError(name)
    X = (X - X.mean(0)) / X.std(0)
    return X, y, len(np.unique(y))

EXTRA = {"seeds": "wheat-seeds.csv", "glass": "glass.csv", "ionosphere": "ionosphere.csv",
         "banknote": "banknote_authentication.csv", "ecoli": "ecoli.csv"}
def real_dataset3(name, data_dir="data"):
    import os, pandas as pd
    if name not in EXTRA:
        return real_dataset2(name, data_dir)
    d = pd.read_csv(os.path.join(data_dir, EXTRA[name]), header=None)
    X = d.iloc[:, :-1].to_numpy(float)
    y = pd.factorize(d.iloc[:, -1].astype(str).str.strip(), sort=True)[0]
    X = X[:, X.std(0) > 0]
    X = (X - X.mean(0)) / X.std(0)
    return X, y.astype(int), len(np.unique(y))

In [ ]:
import time
import numpy as np

def run_it2fcm_nt(X, V0, m1, m2, mc=None, iters=300, tol=1e-6):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds(dist(X, V), m1, m2)
        W = 0.5 * (lo ** mc + up ** mc)
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds(dist(X, V), m1, m2)
    return V, (0.5 * (lo + up)).argmax(1)

def contingency(y, p, C):
    M = np.zeros((int(y.max()) + 1, C)); np.add.at(M, (y, p), 1); return M

def ari_from_cont(M):
    comb = lambda x: x * (x - 1) / 2.0
    n = M.sum(); s_ij = comb(M).sum(); a = comb(M.sum(1)).sum(); b = comb(M.sum(0)).sum()
    exp = a * b / comb(n); mx = 0.5 * (a + b)
    return float((s_ij - exp) / (mx - exp)) if mx != exp else 1.0

def nmi_from_cont(M):
    n = M.sum(); P = M / n; pi = P.sum(1); pj = P.sum(0)
    nz = P > 0
    I = float((P[nz] * np.log(P[nz] / (pi[:, None] * pj[None, :])[nz])).sum())
    H = lambda q: float(-(q[q > 0] * np.log(q[q > 0])).sum())
    d = 0.5 * (H(pi) + H(pj))
    return I / d if d > 0 else 1.0

def acc_from_cont(M):
    from scipy.optimize import linear_sum_assignment
    r, c = linear_sum_assignment(-M); return float(M[r, c].sum() / M.sum())

def scores(y, p, C):
    M = contingency(y, p, C)
    return dict(acc=acc_from_cont(M), ari=ari_from_cont(M), nmi=nmi_from_cont(M))

def silhouette_pre(Dsub, lab):
    ks = np.unique(lab)
    if len(ks) < 2: return -1.0
    n = len(lab); a = np.zeros(n); b = np.full(n, np.inf)
    for k in ks:
        mk = lab == k; cnt = mk.sum()
        s = Dsub[:, mk].sum(1)
        own = lab == k
        a[own] = s[own] / max(cnt - 1, 1)
        other = ~own
        b[other] = np.minimum(b[other], s[other] / cnt)
    sing = np.array([(lab == lab[i]).sum() == 1 for i in range(n)])
    sil = np.where(sing, 0.0, (b - a) / np.maximum(np.maximum(a, b), EPS))
    return float(sil.mean())

def calinski(X, lab):
    ks = np.unique(lab); n, k = len(X), len(ks)
    if k < 2 or k >= n: return -1.0
    mu = X.mean(0); B = W = 0.0
    for c in ks:
        Xc = X[lab == c]; mc_ = Xc.mean(0)
        B += len(Xc) * ((mc_ - mu) ** 2).sum(); W += ((Xc - mc_) ** 2).sum()
    return float((B / (k - 1)) / (W / (n - k))) if W > 0 else -1.0

def noise_instance(frac, seed):
    r = np.random.default_rng(seed); c = np.array([[0.0, 0.0], [0.0, 4.0]]); k = int(round(frac * 100))
    X = np.vstack([r.normal(c[0], 1.0, (100, 2)), r.normal(c[1], 1.0, (100, 2)), r.normal(c[0], 3.0, (k, 2))])
    y = np.r_[np.zeros(100), np.ones(100), np.zeros(k)].astype(int)
    return X, y, 2

def imbalance_instance(C, ratio, seed):
    r = np.random.default_rng(seed)
    R = 3.5 / (2 * np.sin(np.pi / C)); ang = 2 * np.pi * np.arange(C) / C
    c = np.c_[R * np.cos(ang), R * np.sin(ang)]
    n = np.maximum(np.round(100 * ratio ** (-np.arange(C) / max(C - 1, 1))).astype(int), 3)
    X = np.vstack([r.normal(c[k], 1.0, (n[k], 2)) for k in range(C)])
    y = np.concatenate([np.full(n[k], k) for k in range(C)])
    return X, y, C

In [ ]:
from PIL import Image
GRID = (1.05, 1.1, 1.3, 1.5, 2.0, 3.0, 5.0, 10.0, 20.0)
PAIRS = [(a, b) for i, a in enumerate(GRID) for b in GRID[i + 1:]]
IT2_KEYS = [(a, b, mode) for (a, b) in PAIRS for mode in ("m1", "mid", "m2")]
DEC_KEYS = [(a, b) for a in GRID for b in GRID]
FAMILIES = {"FCM": list(GRID), "IT2FCM": IT2_KEYS, "IT2FCM-NT": IT2_KEYS, "T1dec": DEC_KEYS}
SYN = ("equal", "volume", "noise", "density", "volume3")
REAL = ("iris", "wine", "wdbc", "wbc", "sonar", "pima", "seeds", "glass", "ionosphere", "banknote", "ecoli")
NOISE = {f"noise{int(f*100):02d}": f for f in (0.0, 0.1, 0.2, 0.4, 0.6)}
IMB = {f"imb_C{C}_r{r}": (C, r) for C in (3, 5, 8) for r in (1, 5, 20)}
N_SYN, N_REAL, N_SWEEP, N_INIT, TOL, N_SIL = 40, 20, 20, 2, 1e-6, 1000

def get_instance(nm, s, data_dir):
    if nm in SYN:
        X, y, c, C = scenario(nm, 1000 + s); return X, y, C
    if nm in NOISE: return noise_instance(NOISE[nm], 6000 + s)
    if nm in IMB: return imbalance_instance(*IMB[nm], 7000 + s)
    X, y, C = real_dataset3(nm, data_dir); r = np.random.default_rng(3000 + s)
    idx = np.sort(np.concatenate([r.choice(np.where(y == k)[0], int(round(0.75 * (y == k).sum())), replace=False) for k in range(C)]))
    return X[idx], y[idx], C

def fit(fam, key, X, V0, order):
    if fam == "FCM": return run_fcm(X, V0, key, tol=TOL)
    if fam == "T1dec": return run_fcm_decoupled(X, V0, key[0], key[1], tol=TOL)
    a, b, mode = key; mc = {"m1": a, "mid": 0.5 * (a + b), "m2": b}[mode]
    if fam == "IT2FCM":
        V, hp, hn = run_it2fcm_fast(X, V0, a, b, mc=mc, tol=TOL, order=order); return V, hp
    return run_it2fcm_nt(X, V0, a, b, mc=mc, tol=TOL)

def run_instance06(X, y, C, s, evaluate=None):
    order = np.argsort(X, axis=0, kind="stable")
    r = np.random.default_rng(8000 + s); sub = np.sort(r.choice(len(X), min(N_SIL, len(X)), replace=False))
    Dsub = np.linalg.norm(X[sub][:, None] - X[sub][None], axis=2)
    rec = {f: {k: dict(ari=[], acc=[], nmi=[], f=[], sil=[], ch=[]) for k in keys} for f, keys in FAMILIES.items()}
    secs = {f: 0.0 for f in FAMILIES}
    for q in range(N_INIT):
        V0 = init_pp(X, C, np.random.default_rng(10_000 * (q + 1) + s))
        for fam, keys in FAMILIES.items():
            t0 = time.perf_counter()
            for k in keys:
                V, lab = fit(fam, k, X, V0, order)
                sc = evaluate(V, lab) if evaluate else scores(y, lab, C)
                d = rec[fam][k]
                for m in ("ari", "acc", "nmi"): d[m].append(sc[m])
                d["f"].append(sc.get("f", np.nan)); d["sil"].append(silhouette_pre(Dsub, lab[sub])); d["ch"].append(calinski(X, lab))
            secs[fam] += time.perf_counter() - t0
    return dict(rec=rec, sec_per_run={f: secs[f] / (N_INIT * len(FAMILIES[f])) for f in FAMILIES})

def dibco_instance(i, f, g):
    I8 = np.array(Image.open(f).convert("L")); G = np.array(Image.open(g).convert("L")) < 128
    hist_ink = np.bincount(I8[G].ravel(), minlength=256).astype(float)
    hist_bg = np.bincount(I8[~G].ravel(), minlength=256).astype(float)
    r = np.random.default_rng(5000 + i)
    X = (I8.ravel()[r.choice(I8.size, min(2000, I8.size), replace=False)] / 255.0)[:, None]
    def evaluate(V, lab):
        t = float(V.ravel().mean()); k = int(np.clip(np.ceil(t * 255.0), 0, 256))
        tp, fp = hist_ink[:k].sum(), hist_bg[:k].sum(); fn, tn = hist_ink.sum() - tp, hist_bg.sum() - fp
        M = np.array([[tn, fp], [fn, tp]])
        return dict(acc=float((tp + tn) / M.sum()), ari=ari_from_cont(M), nmi=nmi_from_cont(M),
                    f=float(2 * tp / max(2 * tp + fp + fn, 1)))
    out = run_instance06(X, np.zeros(len(X), int), 2, i, evaluate=evaluate)
    out["name"] = os.path.basename(f).replace("TRAIN_", "").replace("_source", "")
    return out

In [ ]:
def hr_bounds_minmax(D, m1, m2):
    U1, U2 = fcm_membership(D, m1), fcm_membership(D, m2)
    return np.minimum(U1, U2), np.maximum(U1, U2)

def run_it2fcm_mm(X, V0, m1, m2, mc=None, iters=300, tol=1e-6, order=None):
    mc = 0.5 * (m1 + m2) if mc is None else mc
    order = np.argsort(X, axis=0, kind="stable") if order is None else order
    V = V0.copy()
    for _ in range(iters):
        lo, up = hr_bounds_minmax(dist(X, V), m1, m2)
        Vn, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
        if np.abs(Vn - V).max() < tol:
            V = Vn; break
        V = Vn
    lo, up = hr_bounds_minmax(dist(X, V), m1, m2)
    _, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
    return V, (0.5 * (uL + uR)).argmax(1), dist(X, V).argmin(1)

MM = "IT2FCM-mm"

def run_instance07(X, y, C, s, evaluate=None):
    order = np.argsort(X, axis=0, kind="stable")
    r = np.random.default_rng(8000 + s); sub = np.sort(r.choice(len(X), min(N_SIL, len(X)), replace=False))
    Dsub = np.linalg.norm(X[sub][:, None] - X[sub][None], axis=2)
    rec = {k: dict(ari=[], acc=[], nmi=[], f=[], sil=[], ch=[]) for k in IT2_KEYS}
    t0 = time.perf_counter(); mism_pts = mism_runs = 0
    for q in range(N_INIT):
        V0 = init_pp(X, C, np.random.default_rng(10_000 * (q + 1) + s))
        for k in IT2_KEYS:
            a, b, mode = k; mc = {"m1": a, "mid": 0.5 * (a + b), "m2": b}[mode]
            V, hp, hn = run_it2fcm_mm(X, V0, a, b, mc=mc, tol=TOL, order=order)
            sc = evaluate(V, hp) if evaluate else scores(y, hp, C)
            d = rec[k]
            for m in ("ari", "acc", "nmi"): d[m].append(sc[m])
            d["f"].append(sc.get("f", np.nan)); d["sil"].append(silhouette_pre(Dsub, hp[sub])); d["ch"].append(calinski(X, hp))
            mm_ = int((hp != hn).sum()); mism_pts += mm_; mism_runs += int(mm_ > 0)
    return dict(rec={MM: rec}, sec_per_run={MM: (time.perf_counter() - t0) / (N_INIT * len(IT2_KEYS))},
                mism_points=mism_pts, mism_runs=mism_runs)

def dibco_instance07(i, f, g):
    I8 = np.array(Image.open(f).convert("L")); G = np.array(Image.open(g).convert("L")) < 128
    hist_ink = np.bincount(I8[G].ravel(), minlength=256).astype(float)
    hist_bg = np.bincount(I8[~G].ravel(), minlength=256).astype(float)
    r = np.random.default_rng(5000 + i)
    X = (I8.ravel()[r.choice(I8.size, min(2000, I8.size), replace=False)] / 255.0)[:, None]
    def evaluate(V, lab):
        t = float(V.ravel().mean()); k = int(np.clip(np.ceil(t * 255.0), 0, 256))
        tp, fp = hist_ink[:k].sum(), hist_bg[:k].sum(); fn, tn = hist_ink.sum() - tp, hist_bg.sum() - fp
        M = np.array([[tn, fp], [fn, tp]])
        return dict(acc=float((tp + tn) / M.sum()), ari=ari_from_cont(M), nmi=nmi_from_cont(M), f=float(2 * tp / max(2 * tp + fp + fn, 1)))
    out = run_instance07(X, np.zeros(len(X), int), 2, i, evaluate=evaluate)
    out["name"] = os.path.basename(f).replace("TRAIN_", "").replace("_source", "")
    return out

In [ ]:
MAXIT = 300
def fcm10(X, V0, m):
    V = V0.copy(); conv = False
    for it in range(1, MAXIT + 1):
        U = fcm_membership(dist(X, V), m); W = U ** m
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < TOL:
            V = Vn; conv = True; break
        V = Vn
    return V, fcm_membership(dist(X, V), m).argmax(1), None, it, conv

def t1d10(X, V0, ma, mb):
    V = V0.copy(); conv = False
    for it in range(1, MAXIT + 1):
        U = fcm_membership(dist(X, V), ma); W = U ** mb
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < TOL:
            V = Vn; conv = True; break
        V = Vn
    return V, dist(X, V).argmin(1), None, it, conv

def it2km10(X, V0, m1, m2, mc, order, bounds):
    V = V0.copy(); conv = False
    for it in range(1, MAXIT + 1):
        lo, up = bounds(dist(X, V), m1, m2)
        Vn, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
        if np.abs(Vn - V).max() < TOL:
            V = Vn; conv = True; break
        V = Vn
    lo, up = bounds(dist(X, V), m1, m2)
    _, uL, uR = it2fcm_update_fast(X, order, lo, up, mc)
    return V, (0.5 * (uL + uR)).argmax(1), dist(X, V).argmin(1), it, conv

def nt10(X, V0, m1, m2, mc):
    V = V0.copy(); conv = False
    for it in range(1, MAXIT + 1):
        lo, up = hr_bounds(dist(X, V), m1, m2)
        W = 0.5 * (lo ** mc + up ** mc)
        Vn = (W.T @ X) / np.maximum(W.sum(0)[:, None], EPS)
        if np.abs(Vn - V).max() < TOL:
            V = Vn; conv = True; break
        V = Vn
    lo, up = hr_bounds(dist(X, V), m1, m2)
    return V, (0.5 * (lo + up)).argmax(1), None, it, conv

def fit10(fam, key, X, V0, order):
    if fam == "FCM": return fcm10(X, V0, key)
    if fam == "T1dec": return t1d10(X, V0, key[0], key[1])
    a, b, mode = key; mc = {"m1": a, "mid": 0.5 * (a + b), "m2": b}[mode]
    if fam == "IT2FCM": return it2km10(X, V0, a, b, mc, order, hr_bounds)
    if fam == MM: return it2km10(X, V0, a, b, mc, order, hr_bounds_minmax)
    return nt10(X, V0, a, b, mc)

def xie_beni(X, V, m=2.0):
    D = dist(X, V); U = fcm_membership(D, m)
    sep = min(float(((V[j] - V[k]) ** 2).sum()) for j in range(len(V)) for k in range(j + 1, len(V)))
    return float(((U ** m) * D ** 2).sum() / (len(X) * max(sep, EPS)))

def keys_for(fam, G):
    if fam == "FCM": return list(G)
    if fam == "T1dec": return [(a, b) for a in G for b in G]
    return [(a, b, mode) for i, a in enumerate(G) for b in G[i + 1:] for mode in ("m1", "mid", "m2")]

def run_instance10(X, y, C, s, fams, centres=None):
    order = np.argsort(X, axis=0, kind="stable")
    r = np.random.default_rng(8000 + s); sub = np.sort(r.choice(len(X), min(N_SIL, len(X)), replace=False))
    Dsub = np.linalg.norm(X[sub][:, None] - X[sub][None], axis=2)
    fields = ["ari", "acc", "nmi", "sil", "ch", "negxb", "it", "conv"] + (["negcerr"] if centres is not None else [])
    rec = {f: {k: {fl: [] for fl in fields + (["acc_n", "ari_n"] if f == MM else [])} for k in keys} for f, keys in fams.items()}
    V_store = {f: {k: [] for k in keys} for f, keys in fams.items()}; secs = {f: 0.0 for f in fams}
    for q in range(N_INIT):
        V0 = init_pp(X, C, np.random.default_rng(10_000 * (q + 1) + s))
        for fam, keys in fams.items():
            t0 = time.perf_counter()
            for k in keys:
                V, lab, near, it, conv = fit10(fam, k, X, V0, order)
                sc = scores(y, lab, C); d = rec[fam][k]
                for fl in ("ari", "acc", "nmi"): d[fl].append(sc[fl])
                d["sil"].append(silhouette_pre(Dsub, lab[sub])); d["ch"].append(calinski(X, lab)); d["negxb"].append(-xie_beni(X, V))
                d["it"].append(it); d["conv"].append(bool(conv))
                if centres is not None: d["negcerr"].append(-centre_error_h(V, centres))
                if fam == MM:
                    sn = scores(y, near, C); d["acc_n"].append(sn["acc"]); d["ari_n"].append(sn["ari"])
                V_store[fam][k].append(V.astype(np.float32))
            secs[fam] += time.perf_counter() - t0
    return dict(rec=rec, V=V_store, C=C, sec_per_run={f: secs[f] / (N_INIT * len(fams[f])) for f in fams})

In [ ]:
DATA_DIR = os.path.join(OUTDIR, "data"); os.makedirs(DATA_DIR, exist_ok=True)
MIRROR = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/"
UCI = "https://archive.ics.uci.edu/ml/machine-learning-databases/"
EXPECT = {"new_thyroid": (215, 5, 3), "dermatology": (358, 34, 6), "yeast": (1484, 8, 10), "page_blocks": (5473, 10, 5)}

def _get(url, fn):
    p = os.path.join(DATA_DIR, fn)
    if not os.path.exists(p) or os.path.getsize(p) == 0:
        urllib.request.urlretrieve(url, p)
    return p

def _parse(name, how, p):
    import pandas as pd
    if name == "new_thyroid" and how == "mirror":
        d = pd.read_csv(p, header=None); X, y = d.iloc[:, :5], d.iloc[:, 5]
    elif name == "new_thyroid":
        d = pd.read_csv(p, header=None); X, y = d.iloc[:, 1:], d.iloc[:, 0]
    elif name == "dermatology":
        d = pd.read_csv(p, header=None, na_values="?").dropna(); X, y = d.iloc[:, :34], d.iloc[:, 34]
    elif name == "yeast":
        d = pd.read_csv(p, header=None, sep=r"\s+"); X, y = d.iloc[:, 1:9], d.iloc[:, 9]
    elif name == "page_blocks":
        d = pd.read_csv(p, header=None, sep=r"\s+"); X, y = d.iloc[:, :10], d.iloc[:, 10]
    return X.to_numpy(float), y.astype(str).str.strip().to_numpy()

def _openml(name):
    import pandas as pd
    from sklearn.datasets import fetch_openml
    b = fetch_openml(name={"page_blocks": "page-blocks", "new_thyroid": "thyroid-new"}.get(name, name), version=1, as_frame=True, parser="auto")
    X = b.data.apply(pd.to_numeric, errors="coerce"); keep = ~X.isna().any(axis=1)
    return X[keep].to_numpy(float), b.target[keep].astype(str).str.strip().to_numpy()

def load_partb(name):
    cache = os.path.join(DATA_DIR, f"partb_{name}.npz")
    if os.path.exists(cache):
        z = np.load(cache, allow_pickle=False); return z["X"], z["y"], int(z["C"])
    tries = {"new_thyroid": [("mirror", MIRROR + "new-thyroid.csv", "new-thyroid.csv"), ("uci", UCI + "thyroid-disease/new-thyroid.data", "new-thyroid.data"), ("openml", None, None)],
             "dermatology": [("uci", UCI + "dermatology/dermatology.data", "dermatology.data"), ("openml", None, None)],
             "yeast": [("uci", UCI + "yeast/yeast.data", "yeast.data"), ("openml", None, None)],
             "page_blocks": [("uciZ", UCI + "page-blocks/page-blocks.data.Z", "page-blocks.data.Z"), ("openml", None, None)]}[name]
    errors = []
    for how, url, fn in tries:
        try:
            if how == "openml":
                Xr, yr = _openml(name)
            elif how == "uciZ":
                pz = _get(url, fn); p = pz[:-2]
                subprocess.run(f"gzip -dc '{pz}' > '{p}'", shell=True, check=True); Xr, yr = _parse(name, how, p)
            else:
                Xr, yr = _parse(name, how, _get(url, fn))
            break
        except Exception as e:
            errors.append(f"{how}: {e!r}"); Xr = None
    if Xr is None:
        raise RuntimeError(f"{name}: every source failed:\n  " + "\n  ".join(errors))
    import pandas as pd
    y = pd.factorize(yr, sort=True)[0].astype(int); C = len(np.unique(y))
    N0, M0, C0 = EXPECT[name]
    assert C == C0 and Xr.shape[1] == M0 and abs(len(Xr) - N0) <= 10, f"{name}: unexpected shape {Xr.shape}, C={C}"
    X = Xr[:, Xr.std(0) > 0]; X = (X - X.mean(0)) / X.std(0)
    np.savez(cache, X=X, y=y, C=C); return X, y, C

In [ ]:
# Document images: decisions of the type-1 and interval memberships.
from PIL import Image
DIBCO_ROOT = os.environ.get("DIBCO_ROOT") or ("/content/dibco_all" if IN_COLAB else os.path.join(OUTDIR, "dibco_all"))
DIBCO_DIR = os.path.join(DIBCO_ROOT, "repo", "SauvolaNet", "Dataset")
if not os.path.isdir(DIBCO_DIR):
    os.makedirs(DIBCO_ROOT, exist_ok=True); print("cloning DIBCO collection (~945 MB) ...")
    subprocess.run(["git", "clone", "--depth", "1", "-q", "https://github.com/RichSu95/Document_Binarization_Collection.git",
                    os.path.join(DIBCO_ROOT, "repo")], check=True)
EPS02 = 1e-9; M1_02, M2_02, MT1_02, ITERS02 = 1.1, 10.1, 2.0, 30
def load02(imgf, gtf):
    I = np.array(Image.open(imgf).convert("L"), dtype=float) / 255.0
    return I, np.array(Image.open(gtf).convert("L")) < 128
def fcm_u02(I, v1, v2, m):
    d1 = np.abs(I - v1) + EPS02; d2 = np.abs(I - v2) + EPS02
    return 1.0 / (1.0 + np.clip((d1 / d2) ** (2.0 / (m - 1.0)), 0, 1e12))
def fit02(I, m=MT1_02, iters=ITERS02):
    v1, v2 = np.percentile(I, 5), np.percentile(I, 95)
    for _ in range(iters):
        u = fcm_u02(I, v1, v2, m); w1, w2 = u ** m, (1 - u) ** m
        v1n = (w1 * I).sum() / max(w1.sum(), EPS02); v2n = (w2 * I).sum() / max(w2.sum(), EPS02)
        if abs(v1n - v1) < 1e-6 and abs(v2n - v2) < 1e-6:
            v1, v2 = v1n, v2n; break
        v1, v2 = v1n, v2n
    return float(v1), float(v2)
def f_measure02(pred, gt):
    tp = float((pred & gt).sum()); p = tp / max(pred.sum(), 1); r = tp / max(gt.sum(), 1)
    return 2 * p * r / max(p + r, EPS02)
PAIRS02 = sorted((f, f.replace("_source.", "_target.")) for f in glob.glob(os.path.join(DIBCO_DIR, "*DIBCO*_source.*")))
PAIRS02 = [(a, b) for a, b in PAIRS02 if os.path.exists(b)]
print(len(PAIRS02), "DIBCO image pairs")
rows02 = []; total_xor = 0
for f, g in PAIRS02:
    I, G = load02(f, g); v1, v2 = fit02(I)
    a, b = fcm_u02(I, v1, v2, M1_02), fcm_u02(I, v1, v2, M2_02); u2 = 0.5 * (np.minimum(a, b) + np.maximum(a, b))
    u1 = fcm_u02(I, v1, v2, MT1_02); xor = (u2 > 0.5) ^ (u1 > 0.5); total_xor += int(xor.sum())
    key = os.path.splitext(os.path.basename(f).replace("TRAIN_", "").replace("_source", ""))[0]
    rows02.append(dict(img=key, f=f, g=g, v1=v1, v2=v2, f_t1=f_measure02(u1 > 0.5, G), diff_pixels=int(xor.sum())))
print(f"differing pixels over all {len(rows02)} images: {total_xor}")
by_f = sorted(rows02, key=lambda r: r["f_t1"])
picks = [by_f[-1], by_f[len(by_f) // 2], by_f[max(0, len(by_f) // 6)]]
dmax = 0.0
for rec in rows02:
    I, _ = load02(rec["f"], rec["g"])
    a, b = fcm_u02(I, rec["v1"], rec["v2"], M1_02), fcm_u02(I, rec["v1"], rec["v2"], M2_02)
    u2 = 0.5 * (np.minimum(a, b) + np.maximum(a, b)); dmax = max(dmax, float(np.abs(u2 - fcm_u02(I, rec["v1"], rec["v2"], MT1_02)).max()))
for rec in picks:
    I, G = load02(rec["f"], rec["g"])
    a, b = fcm_u02(I, rec["v1"], rec["v2"], M1_02), fcm_u02(I, rec["v1"], rec["v2"], M2_02)
    u2 = 0.5 * (np.minimum(a, b) + np.maximum(a, b)); u1 = fcm_u02(I, rec["v1"], rec["v2"], MT1_02)
    xor = (u2 > 0.5) ^ (u1 > 0.5); d = np.abs(u2 - u1)
    LOG.setdefault("dibco", []).append(dict(image=rec["img"], F_type1=rec["f_t1"], max_abs_diff=float(d.max()), differing_pixels=int(xor.sum())))
LOG["dibco_all"] = dict(images=len(rows02), differing_pixels_total=total_xor, max_abs_diff=dmax)

In [ ]:
# Largest class kept in one cluster and clusters left empty, at the selected settings.
for _f in ("ecoli.csv", "glass.csv"): _get(MIRROR + _f, _f)
from scipy.optimize import linear_sum_assignment
import ast
def parse(sel): return ast.literal_eval(sel) if isinstance(sel, str) and sel.startswith("(") else float(sel)
def selection(nm, split="A->B"):
    metric = "ari" if nm in IMB or nm in NOISE else "acc"
    sp = J08["results"][metric][nm]["label_tuned"]["splits"][split]["selected"]
    return {f: parse(sp[f]) for f in ("FCM", "T1dec", MM)}, metric
def matched(y, lab, C):
    M = np.zeros((C, C), int); np.add.at(M, (y, lab), 1); r, c = linear_sum_assignment(-M)
    to_class = np.empty(C, int); to_class[c] = r; return to_class[lab], M[:, c[np.argsort(r)]]
def describe(X, y, C, lab):
    sz = np.bincount(lab, minlength=C); big = np.argmax(np.bincount(y, minlength=C))
    return dict(tiny=int((sz <= max(2, 0.01 * len(X))).sum()), kept=float(np.bincount(lab[y == big], minlength=C).max() / (y == big).sum()))
MECH = {}
for nm in ("imb_C3_r20", "imb_C5_r20", "imb_C8_r20", "imb_C8_r1", "ecoli", "glass"):
    st = {f: dict(tiny=[], kept=[], score=[]) for f in ("FCM", "T1dec", MM)}
    n = N_SWEEP if nm in IMB else N_REAL
    for sn, half in (("A->B", range(n // 2, n)), ("B->A", range(0, n // 2))):
        sel, metric = selection(nm, sn)
        for s in half:
            X, y, C = get_instance(nm, s, DATA_DIR); order = np.argsort(X, 0, kind="stable")
            for q in range(N_INIT):
                V0 = init_pp(X, C, np.random.default_rng(10_000 * (q + 1) + s))
                for f in st:
                    V, lab, _, _, _ = fit10(f, sel[f], X, V0, order); d = describe(X, y, C, lab)
                    st[f]["tiny"].append(d["tiny"]); st[f]["kept"].append(d["kept"]); st[f]["score"].append(scores(y, lab, C)[metric])
    MECH[nm] = {f: dict(empty_or_tiny_clusters=float(np.mean(v["tiny"])), largest_class_kept=float(np.mean(v["kept"])),
                        metric=metric, score=float(np.mean(v["score"]))) for f, v in st.items()}
    print(nm, " | ".join(f"{f.replace('IT2FCM-mm','IT2').replace('T1dec','T1d')}: tiny {MECH[nm][f]['empty_or_tiny_clusters']:.2f}, "
                         f"largest kept {MECH[nm][f]['largest_class_kept']:.2f}, {metric} {MECH[nm][f]['score']:.3f}" for f in st), flush=True)
LOG["regime_mechanism"] = MECH
def representative(nm):
    sel, metric = selection(nm); n = N_SWEEP if nm in IMB else N_REAL; res = {}
    for s in range(n // 2, n):
        X, y, C = get_instance(nm, s, DATA_DIR); order = np.argsort(X, 0, kind="stable"); runs = []
        for q in range(N_INIT):
            V0 = init_pp(X, C, np.random.default_rng(10_000 * (q + 1) + s))
            runs.append({f: fit10(f, sel[f], X, V0, order)[:2] for f in ("FCM", "T1dec", MM)})
        res[s] = (X, y, C, runs)
    g = {s: np.mean([scores(res[s][1], r[MM][1], res[s][2])[metric] - scores(res[s][1], r["FCM"][1], res[s][2])[metric] for r in res[s][3]]) for s in res}
    s_med = sorted(g, key=g.get)[(len(g) - 1) // 2]; X, y, C, runs = res[s_med]
    q = int(np.argmin([abs(scores(y, r[MM][1], C)[metric] - scores(y, r["FCM"][1], C)[metric] - g[s_med]) for r in runs]))
    return X, y, C, runs[q], metric, sel, s_med, q, g
lab_ = {"FCM": "FCM", "T1dec": "T1d", MM: "IT2"}
X, y, C, r, metric, sel, s_med, q, g = representative("imb_C8_r20")
LOG["regime_row1"] = dict(set="imb_C8_r20", instance=s_med, init=q, settings={k: str(v) for k, v in sel.items()},
                         shown={lab_[f]: dict(ari=scores(y, r[f][1], C)["ari"], **describe(X, y, C, r[f][1])) for f in r})
X, y, C, r, metric, sel, s_med, q, g = representative("ecoli")
LOG["regime_row2"] = dict(set="ecoli", instance=s_med, init=q, settings={k: str(v) for k, v in sel.items()},
                         shown={lab_[f]: dict(acc=scores(y, r[f][1], C)["acc"], **describe(X, y, C, r[f][1])) for f in r})

In [ ]:
# Natural images: footprint widths near and away from the boundary.
from scipy.io import loadmat
BROOT = os.environ.get("BSDS_ROOT") or ("/content/bsds" if IN_COLAB else os.path.join(OUTDIR, "bsds"))
BDATA = os.path.join(BROOT, "repo", "BSDS500", "data")
if not os.path.isdir(BDATA):
    os.makedirs(BROOT, exist_ok=True); print("cloning BSDS500 (~390 MB) ...")
    subprocess.run(["git", "clone", "--depth", "1", "-q", "https://github.com/BIDS/BSDS500.git", os.path.join(BROOT, "repo")], check=True)
def bsds_u(g, m=2.0, iters=40):
    v = np.array([np.quantile(g, 0.25), np.quantile(g, 0.75)])
    for _ in range(iters):
        d = np.abs(g[None, :] - v[:, None]) + 1e-12; w = d ** (-2.0 / (m - 1.0)); u = w / w.sum(0, keepdims=True)
        um = u ** m; v = (um * g[None, :]).sum(1) / um.sum(1)
    return u[int(np.argmax(v))]
def at_m(u, m):
    g_ = 1.0 / (m - 1.0); u = np.clip(u, 1e-12, 1 - 1e-12); return u ** g_ / (u ** g_ + (1 - u) ** g_)
files = sorted(glob.glob(os.path.join(BDATA, "images", "test", "*.jpg"))); cand = []
for f in files:
    p = os.path.join(BDATA, "groundTruth", "test", os.path.basename(f)[:-4] + ".mat")
    if not os.path.exists(p): continue
    gt = loadmat(p)["groundTruth"][0]; masks = [np.asarray(g_["Boundaries"][0, 0], dtype=float) for g_ in gt]
    if len(masks) < 4: continue
    frac = np.stack(masks).mean(0); Y = (frac > 0) & (frac < 1)
    if Y.sum() < 500 or (~Y).sum() < 500: continue
    cand.append((f, Y, float(Y.mean())))
    if len(cand) >= 60: break
cand.sort(key=lambda c: c[2]); picks_b = [cand[0], cand[len(cand) // 2], cand[-1]]
for i, (f, Y, fr) in enumerate(picks_b):
    img = np.asarray(Image.open(f).convert("L"), dtype=float) / 255.0
    u = bsds_u(img.ravel()).reshape(img.shape)
    a1, a2 = at_m(u, 1.1), at_m(u, 10.1); w_pair = np.abs(a1 - a2); w_pow = np.clip(u, 0, 1) ** 0.85 - np.clip(u, 0, 1) ** 1.15
    near = np.abs(u - 0.5) < 0.05
    LOG.setdefault("bsds", []).append(dict(image=os.path.basename(f), disagreement_fraction=fr,
        pair_width_near_boundary=float(w_pair[near].mean()) if near.any() else None, pair_width_elsewhere=float(w_pair[~near].mean()),
        power_width_near_boundary=float(w_pow[near].mean()) if near.any() else None, power_width_elsewhere=float(w_pow[~near].mean())))

In [ ]:
def sel_cerr(nm, split="A->B"):
    sp = J10["analysis"]["A1"][nm]["negcerr"]["label_tuned"]["splits"][split]["selected"]
    return {f: (ast.literal_eval(sp[f]) if sp[f].startswith("(") else float(sp[f])) for f in ("FCM", "T1dec", "IT2FCM", MM)}
def centres_syn(nm, s): return scenario(nm, 1000 + s)[2]
HRS = [("equal", "Equal volumes"), ("volume", "Unequal volumes"), ("noise", "Noise"), ("density", "Unequal sizes"), ("volume3", "Unequal volumes, C=3")]
lab_ = {"FCM": "FCM", "T1dec": "T1d", "IT2FCM": "IT2p", MM: "IT2"}
for nm, title in HRS:
    sel = sel_cerr(nm); n = N_SYN; res = {}
    for s in range(n // 2, n):
        X, y, C = get_instance(nm, s, DATA_DIR); order = np.argsort(X, 0, kind="stable"); c = centres_syn(nm, s); runs = []
        for q in range(N_INIT):
            V0 = init_pp(X, C, np.random.default_rng(10_000 * (q + 1) + s))
            runs.append({f: fit10(f, sel[f], X, V0, order)[0] for f in sel})
        res[s] = (X, y, C, c, runs)
    g = {s: np.mean([centre_error_h(r[MM], res[s][3]) - centre_error_h(r["FCM"], res[s][3]) for r in res[s][4]]) for s in res}
    s_med = sorted(g, key=g.get)[(len(g) - 1) // 2]; X, y, C, c, runs = res[s_med]
    q = int(np.argmin([abs(centre_error_h(r[MM], c) - centre_error_h(r["FCM"], c) - g[s_med]) for r in runs])); r = runs[q]
    errs = {lab_[f]: centre_error_h(r[f], c) for f in ("FCM", "T1dec", "IT2FCM", MM)}
    LOG.setdefault("hr_scenarios", []).append(dict(set=nm, instance=s_med, init=q, settings={lab_[k]: str(v) for k, v in sel.items()}, center_error=errs))

In [ ]:
LOG["run_stamp"] = RUN_STAMP; print("=" * 80); print(f"11 -- mechanism where classes differ strongly in size   (run {RUN_STAMP})"); print("=" * 80)
d = LOG.get("dibco_all", {}); print(f"DIBCO: {d.get('images')} images, differing pixels in total {d.get('differing_pixels_total')}")
for r in LOG.get("dibco", []): print(f"   shown {r['image']:14s} F {r['F_type1']:.3f}  max |diff| {r['max_abs_diff']:.2f}  differing pixels {r['differing_pixels']}")
for nm, d in LOG.get("regime_mechanism", {}).items():
    print(f"   {nm:11s} " + " | ".join(f"{k.replace('IT2FCM-mm','IT2').replace('T1dec','T1d')} {v['empty_or_tiny_clusters']:.2f} / {v['largest_class_kept']:.2f} / {v['score']:.3f}" for k, v in d.items()))
for key in ("regime_row1", "regime_row2"):
    r = LOG.get(key)
    if r: print(f"shown {r['set']} instance {r['instance']} init {r['init']}: {r['shown']}\n   settings {r['settings']}")
for r in LOG.get("hr_scenarios", []): print(f"scenario {r['set']:8s} instance {r['instance']} init {r['init']}: " + ", ".join(f"{k} {v:.3f}" for k, v in r["center_error"].items()))
for r in LOG.get("bsds", []): print(f"BSDS {r['image']:12s} disagreement {r['disagreement_fraction']*100:.1f}%  pair width near/elsewhere {r['pair_width_near_boundary']} / {r['pair_width_elsewhere']:.3f}")
p = os.path.join(OUTDIR, "results_11.json")
with open(p, "w") as fh:
    json.dump(LOG, fh, indent=1, default=str); fh.flush(); os.fsync(fh.fileno())
print("\nwritten:", p)